# Social Media Capstone — Full Lakehouse Pipeline (Local PySpark)

**Path B:** Bronze → Silver → Gold → Insights  
**Domain:** Social Media brand sentiment & engagement

## 0. Setup Spark

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import shutil

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BRONZE_DIR = PROJECT_ROOT / 'data' / 'bronze'
SILVER_DIR = PROJECT_ROOT / 'data' / 'silver'
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold'
CHARTS_DIR = PROJECT_ROOT / 'docs' / 'charts'
RAW_CSV = BRONZE_DIR / 'social_media_tweets_raw.csv'

spark = (
    SparkSession.builder.master('local[*]')
    .appName('social-media-capstone')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
print('Project root:', PROJECT_ROOT)

## 1. Bronze — Raw ingestion (no cleaning)

In [ ]:
bronze_df = (
    spark.read.option('header', True)
    .option('multiLine', True)
    .option('quote', '"')
    .option('escape', '"')
    .csv(str(RAW_CSV))
)
print('Rows:', bronze_df.count())
print('Columns:', len(bronze_df.columns))
bronze_df.printSchema()
bronze_df.show(5, truncate=40)

bronze_path = BRONZE_DIR / 'tweets_bronze'
if bronze_path.exists():
    shutil.rmtree(bronze_path)
bronze_df.write.mode('overwrite').parquet(str(bronze_path))
print('Bronze saved →', bronze_path)

## 2. Silver — Cleaning & schema correction

In [ ]:
bronze = spark.read.parquet(str(BRONZE_DIR / 'tweets_bronze'))
before = bronze.count()

silver = bronze.select(
    F.col('tweet_id').alias('post_id'),
    F.lower(F.trim(F.col('airline_sentiment'))).alias('sentiment'),
    F.col('airline_sentiment_confidence').cast('double').alias('sentiment_confidence'),
    F.col('negativereason').alias('negative_reason'),
    F.col('negativereason_confidence').cast('double').alias('negative_reason_confidence'),
    F.col('airline').alias('brand'),
    F.col('name').alias('username'),
    F.col('retweet_count').cast('int').alias('retweet_count'),
    F.col('text').alias('post_text'),
    F.col('tweet_created').alias('posted_at_raw'),
    F.col('tweet_location').alias('location'),
    F.col('user_timezone').alias('user_timezone'),
)

silver = silver.withColumn(
    'posted_at', F.to_timestamp(F.col('posted_at_raw'), 'yyyy-MM-dd HH:mm:ss Z')
)
silver = (
    silver.withColumn('post_date', F.to_date('posted_at'))
    .withColumn('post_hour', F.hour('posted_at'))
    .withColumn('day_of_week', F.date_format('posted_at', 'EEEE'))
    .withColumn('text_length', F.length('post_text'))
)

silver = silver.filter(F.col('post_id').isNotNull())
silver = silver.filter(F.col('post_text').isNotNull() & (F.trim(F.col('post_text')) != ''))
silver = silver.filter(F.col('sentiment').isin('positive', 'negative', 'neutral'))
silver = silver.filter((F.col('retweet_count').isNull()) | (F.col('retweet_count') >= 0))
silver = silver.orderBy(F.col('sentiment_confidence').desc_nulls_last()).dropDuplicates(['post_id'])
silver = silver.fillna({
    'retweet_count': 0,
    'location': 'Unknown',
    'user_timezone': 'Unknown',
    'brand': 'Unknown',
    'username': 'unknown_user',
})

after = silver.count()
print(f'Before: {before:,} | After: {after:,} | Removed: {before - after:,}')
silver.groupBy('sentiment').count().orderBy(F.desc('count')).show()

silver_path = SILVER_DIR / 'tweets_silver'
if silver_path.exists():
    shutil.rmtree(silver_path)
SILVER_DIR.mkdir(parents=True, exist_ok=True)
silver.write.mode('overwrite').parquet(str(silver_path))
print('Silver saved →', silver_path)

## 3. Gold — Business tables

In [ ]:
silver = spark.read.parquet(str(SILVER_DIR / 'tweets_silver'))
GOLD_DIR.mkdir(parents=True, exist_ok=True)

def write_gold(df, name):
    path = GOLD_DIR / name
    if path.exists():
        shutil.rmtree(path)
    df.write.mode('overwrite').parquet(str(path))
    print(name, '→', df.count(), 'rows')

brand_totals = silver.groupBy('brand').agg(F.count('*').alias('total_posts'))
brand_neg = silver.filter(F.col('sentiment') == 'negative').groupBy('brand').agg(F.count('*').alias('negative_posts'))
brand_pos = silver.filter(F.col('sentiment') == 'positive').groupBy('brand').agg(F.count('*').alias('positive_posts'))

brand_scoreboard = (
    brand_totals.join(brand_neg, 'brand', 'left')
    .join(brand_pos, 'brand', 'left')
    .fillna(0, subset=['negative_posts', 'positive_posts'])
    .withColumn('negative_pct', F.round(100 * F.col('negative_posts') / F.col('total_posts'), 2))
    .withColumn('positive_pct', F.round(100 * F.col('positive_posts') / F.col('total_posts'), 2))
    .orderBy(F.desc('negative_pct'))
)
write_gold(brand_scoreboard, 'brand_scoreboard')
brand_scoreboard.show()

negative_reasons = (
    silver.filter((F.col('sentiment') == 'negative') & F.col('negative_reason').isNotNull() & (F.trim(F.col('negative_reason')) != ''))
    .groupBy('negative_reason')
    .agg(F.count('*').alias('complaint_count'))
    .orderBy(F.desc('complaint_count'))
)
write_gold(negative_reasons, 'top_negative_reasons')
negative_reasons.show(10)

activity_by_day = (
    silver.groupBy('day_of_week')
    .agg(F.count('*').alias('post_count'))
    .orderBy(F.desc('post_count'))
)
write_gold(activity_by_day, 'activity_by_day')
activity_by_day.show()

## 4. Insights & charts

In [ ]:
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

scoreboard = spark.read.parquet(str(GOLD_DIR / 'brand_scoreboard')).toPandas()
reasons = spark.read.parquet(str(GOLD_DIR / 'top_negative_reasons')).toPandas()
by_day = spark.read.parquet(str(GOLD_DIR / 'activity_by_day')).toPandas()

print('Worst brand by negative %:')
print(scoreboard.sort_values('negative_pct', ascending=False).head(1))
print('\nTop complaint reason:')
print(reasons.head(1))

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = scoreboard.sort_values('negative_pct', ascending=True)
ax.barh(plot_df['brand'], plot_df['negative_pct'], color='#c0392b')
ax.set_xlabel('Negative posts (%)')
ax.set_title('Social Media Brand Health — % Negative Sentiment')
fig.tight_layout()
fig.savefig(CHARTS_DIR / '01_brand_negative_pct.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
top10 = reasons.head(10)
ax.barh(top10['negative_reason'][::-1], top10['complaint_count'][::-1], color='#2980b9')
ax.set_title('Top Negative Reasons')
fig.tight_layout()
fig.savefig(CHARTS_DIR / '02_top_negative_reasons.png', dpi=150)
plt.show()

print('Charts saved to', CHARTS_DIR)

In [ ]:
spark.stop()
print('Done.')